# 02 - Anomaly Detection Evaluation

This notebook evaluates the anomaly detection system using synthetic anomaly injection.

## Contents
1. Load data and ground truth
2. Inject synthetic anomalies
3. Run detection on modified data
4. Compute evaluation metrics (Precision, Recall, F1, MTTD)
5. Generate evaluation plots

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import timedelta

from src.config import (
    DATA_PROCESSED, METRIC_FACTS_FILE, ANOMALIES_FILE, 
    GROUND_TRUTH_FILE, OUTPUTS_FIGURES
)
from src.detect import run_detection, detect_anomalies_for_series

## 1. Load Data

In [ ]:
# Load metric facts
metric_facts = pd.read_parquet(DATA_PROCESSED / METRIC_FACTS_FILE)
metric_facts['date'] = pd.to_datetime(metric_facts['date'])
print(f"Loaded metric_facts: {len(metric_facts):,} rows")

# Load ground truth
gt_path = DATA_PROCESSED / GROUND_TRUTH_FILE
if gt_path.exists():
    ground_truth = pd.read_parquet(gt_path)
    print(f"Loaded ground_truth: {len(ground_truth)} events")
else:
    print("Ground truth not found. Creating...")
    from src.detect import create_ground_truth_events
    ground_truth = create_ground_truth_events()

# Load detected anomalies
anomalies_path = DATA_PROCESSED / ANOMALIES_FILE
if anomalies_path.exists():
    detected_anomalies = pd.read_parquet(anomalies_path)
    print(f"Loaded detected anomalies: {len(detected_anomalies)} events")
else:
    detected_anomalies = pd.DataFrame()
    print("No detected anomalies found.")

## 2. Anomaly Injection

We inject known anomalies into the data to create a labeled evaluation set.

In [ ]:
def inject_anomaly(df, metric_name, level, segment_key, start_date, duration_days, magnitude, direction):
    """
    Inject a synthetic anomaly into the metric_facts data.
    
    Args:
        df: metric_facts DataFrame
        metric_name: Target metric
        level: Hierarchy level
        segment_key: Segment identifier
        start_date: Anomaly start date
        duration_days: Number of days
        magnitude: Percentage change (0.3 = 30%)
        direction: 'spike' or 'drop'
    
    Returns:
        Modified DataFrame
    """
    df = df.copy()
    
    # Parse segment key
    if segment_key == 'global':
        mask = (df['metric_name'] == metric_name) & (df['level'] == level)
    else:
        seg_level, seg_value = segment_key.split('=')
        col_map = {'state': 'state_id', 'store': 'store_id', 'category': 'cat_id', 'department': 'dept_id'}
        col = col_map.get(seg_level)
        mask = (df['metric_name'] == metric_name) & (df['level'] == level) & (df[col] == seg_value)
    
    # Date range
    end_date = start_date + timedelta(days=duration_days - 1)
    date_mask = (df['date'] >= start_date) & (df['date'] <= end_date)
    
    full_mask = mask & date_mask
    
    # Apply modification
    multiplier = (1 + magnitude) if direction == 'spike' else (1 - magnitude)
    df.loc[full_mask, 'value'] = df.loc[full_mask, 'value'] * multiplier
    
    print(f"Injected {direction} ({magnitude*100:.0f}%) in {metric_name} at {level}:{segment_key}")
    print(f"  Date range: {start_date.date()} to {end_date.date()} ({duration_days} days)")
    print(f"  Affected rows: {full_mask.sum()}")
    
    return df

In [ ]:
# Define injection scenarios
injections = [
    {
        'metric_name': 'revenue',
        'level': 'store',
        'segment_key': 'store=CA_1',
        'start_date': pd.Timestamp('2016-03-15'),
        'duration_days': 3,
        'magnitude': 0.30,
        'direction': 'drop'
    },
    {
        'metric_name': 'units',
        'level': 'global',
        'segment_key': 'global',
        'start_date': pd.Timestamp('2016-04-10'),
        'duration_days': 2,
        'magnitude': 0.25,
        'direction': 'spike'
    },
    {
        'metric_name': 'avg_price',
        'level': 'state',
        'segment_key': 'state=TX',
        'start_date': pd.Timestamp('2016-05-01'),
        'duration_days': 2,
        'magnitude': 0.20,
        'direction': 'spike'
    },
]

# Apply injections
modified_facts = metric_facts.copy()
for inj in injections:
    modified_facts = inject_anomaly(modified_facts, **inj)

## 3. Run Detection on Modified Data

In [ ]:
# Run detection on modified data
print("Running anomaly detection on modified data...")
eval_anomalies = run_detection(modified_facts)
print(f"\nDetected {len(eval_anomalies)} anomaly events")

## 4. Evaluation Metrics

In [ ]:
def evaluate_detection(detected, ground_truth_events, tolerance_days=3):
    """
    Evaluate detection performance against ground truth.
    
    Args:
        detected: DataFrame of detected anomalies
        ground_truth_events: List of injection dictionaries
        tolerance_days: Days tolerance for matching
    
    Returns:
        Dictionary of metrics
    """
    if len(detected) == 0:
        return {'precision': 0, 'recall': 0, 'f1': 0, 'mttd': None, 'fp_per_30d': 0}
    
    true_positives = 0
    detection_delays = []
    matched_gt = set()
    
    for _, det in detected.iterrows():
        det_date = pd.Timestamp(det['peak_date'])
        det_metric = det['metric_name']
        det_segment = det['segment_key']
        
        for i, gt in enumerate(ground_truth_events):
            if i in matched_gt:
                continue
            
            gt_start = gt['start_date']
            gt_end = gt_start + timedelta(days=gt['duration_days'] - 1)
            
            # Check if detection matches ground truth
            date_match = (det_date >= gt_start - timedelta(days=tolerance_days)) and \
                        (det_date <= gt_end + timedelta(days=tolerance_days))
            metric_match = det_metric == gt['metric_name']
            segment_match = det_segment == gt['segment_key']
            
            if date_match and metric_match and segment_match:
                true_positives += 1
                matched_gt.add(i)
                
                # Calculate detection delay
                delay = (det_date - gt_start).days
                detection_delays.append(max(0, delay))
                break
    
    # Calculate metrics
    n_detected = len(detected)
    n_ground_truth = len(ground_truth_events)
    false_positives = n_detected - true_positives
    
    precision = true_positives / n_detected if n_detected > 0 else 0
    recall = true_positives / n_ground_truth if n_ground_truth > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    mttd = np.mean(detection_delays) if detection_delays else None
    
    # Estimate FP per 30 days (based on data span)
    date_range = (detected['peak_date'].max() - detected['peak_date'].min()).days
    fp_per_30d = false_positives * 30 / max(date_range, 1)
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'mttd': mttd,
        'fp_per_30d': fp_per_30d,
        'true_positives': true_positives,
        'false_positives': false_positives,
        'n_ground_truth': n_ground_truth
    }

In [ ]:
# Evaluate
metrics = evaluate_detection(eval_anomalies, injections)

print("=== Evaluation Results ===")
print(f"\nGround Truth Events: {metrics['n_ground_truth']}")
print(f"Detected Events: {len(eval_anomalies)}")
print(f"True Positives: {metrics['true_positives']}")
print(f"False Positives: {metrics['false_positives']}")
print(f"\nPrecision: {metrics['precision']:.2%}")
print(f"Recall: {metrics['recall']:.2%}")
print(f"F1 Score: {metrics['f1']:.2%}")
print(f"\nMean Time to Detect (MTTD): {metrics['mttd']:.1f} days" if metrics['mttd'] else "MTTD: N/A")
print(f"False Positives per 30 days: {metrics['fp_per_30d']:.1f}")

## 5. Evaluation Plots

In [ ]:
# Create metrics summary bar chart
metrics_df = pd.DataFrame({
    'Metric': ['Precision', 'Recall', 'F1 Score'],
    'Value': [metrics['precision'], metrics['recall'], metrics['f1']]
})

fig = px.bar(
    metrics_df,
    x='Metric',
    y='Value',
    title='Detection Performance Metrics',
    text='Value'
)
fig.update_traces(texttemplate='%{text:.1%}', textposition='outside')
fig.update_layout(yaxis_range=[0, 1.1], yaxis_title='Score')
fig.show()

# Save figure
OUTPUTS_FIGURES.mkdir(parents=True, exist_ok=True)
fig.write_html(OUTPUTS_FIGURES / 'detection_metrics.html')
print(f"Saved to {OUTPUTS_FIGURES / 'detection_metrics.html'}")

In [ ]:
# Severity distribution of detected anomalies
if len(eval_anomalies) > 0 and 'z_severity_peak' in eval_anomalies.columns:
    fig = px.histogram(
        eval_anomalies,
        x='z_severity_peak',
        nbins=20,
        title='Distribution of Anomaly Severity Scores'
    )
    fig.update_layout(xaxis_title='Z-Severity', yaxis_title='Count')
    fig.show()
    
    fig.write_html(OUTPUTS_FIGURES / 'severity_distribution.html')

## Conclusion

This notebook evaluated the anomaly detection system using synthetic injections.

Key findings:
- Precision, Recall, and F1 scores indicate detection accuracy
- MTTD shows how quickly anomalies are detected
- False positive rate indicates noise level

Proceed to `03_rca_examples.ipynb` for root cause analysis examples.